# DS 208 · Week 7 — pandas II
## Grouping, merging and reshaping

Every code block from the lecture, in order, so you can run what you watched instead of
retyping it. The first slides use small tables typed into the cells; the **Case study**
cell loads the flood control data from the course API, so there is no file to download.

**How to use this.** Run the cells top to bottom the first time (Runtime → Run all). The
exercise cells list their tasks above the code: predict first, then run, then change one
thing and run again.

**Version differences.** The slides follow pandas 3. On pandas 2, text columns show as
`object` instead of `str`, and dates as `datetime64[ns]` instead of `datetime64[us]`.


In [ ]:
import numpy as np
import pandas as pd

---
## The lecture, cell by cell

Headings carry the slide numbers shown in the deck's counter.


### Slide 7 — Code on these slides runs in the browser


Expected output:

```
pandas version: 3.0.5
64-bit Python: True
city      str
pop     int64
dtype: object
```


In [ ]:
import io, sys

print("pandas version:", pd.__version__)
print("64-bit Python:", sys.maxsize > 2**32)

t = pd.read_csv(io.StringIO("city,pop\nCebu,964000"))
print(t.dtypes)

### Slide 10 — Eight cities in three regions


Expected output:

```
city    region      pop
0     Quezon City     Luzon  2960000
1          Manila     Luzon  1847000
2          Baguio     Luzon   366000
3            Cebu   Visayas   964000
4          Iloilo   Visayas   457000
5        Tacloban   Visayas   251000
6           Davao  Mindanao  1777000
7  Cagayan de Oro  Mindanao   728000
```


In [ ]:
import io

csv_text = """city,region,pop
Quezon City,Luzon,2960000
Manila,Luzon,1847000
Baguio,Luzon,366000
Cebu,Visayas,964000
Iloilo,Visayas,457000
Tacloban,Visayas,251000
Davao,Mindanao,1777000
Cagayan de Oro,Mindanao,728000"""
cities = pd.read_csv(io.StringIO(csv_text))
print(cities)

### Slide 12 — groupby followed by a summary


Expected output:

```
region
Luzon       1724333.0
Mindanao    1252500.0
Visayas      557333.0
Name: pop, dtype: float64
```


In [ ]:
# cities: the table from the start of Part A
print(cities.groupby("region")["pop"].mean().round())

### Slide 13 — .agg() for a per-group report


**Exercise.**

1. Predict which region has the most cities, and which has the largest single city.
2. Run the cell.
3. Change "max" to "median". For which region are the mean and the median equal, and why?


In [ ]:
# cities: the table from the start of Part A
report = cities.groupby("region")["pop"].agg(
    ["mean", "max", "count"])
print(report.round())

### Slide 14 — The flood data, loaded from the course API


Expected output:

```
(34079, 15)
```


In [ ]:
import io, requests

url  = "https://portal.latarak.com/api/flood/export.csv"
resp = requests.get(url, timeout=60)
df   = pd.read_csv(io.StringIO(resp.text))

print(df.shape)

### Slide 15 — Seven statistics per group in one call


Expected output:

```
(18, 7)
['Central Office', 'Region XIII', 'Region XI']
```


In [ ]:
stats = df.groupby("region").budget.agg([
    "count", "sum", "mean", "median",
    "std", "min", "max"])
print(stats.shape)

# sort by whichever column matters
top = stats.sort_values("median", ascending=False)
print(top.index[:3].tolist())

### Slide 16 — Split, apply, combine on the project data


Expected output:

```
34079 projects -> 18 regions
```


In [ ]:
by_region = df.groupby("region").agg(
    projects=("contract_id", "count"),
    total=("budget", "sum"),
    typical=("budget", "median"),
)
print(len(df), "projects ->", len(by_region), "regions")

### Slide 18 — Grouping by more than one column


Expected output:

```
180
198
```


In [ ]:
g = (df.groupby(["region", "year"])
       .budget.sum().reset_index())
print(len(g))
print(18 * 11)       # the full region-year grid

### Slide 19 — Keep the group key as a column


Expected output:

```
Series ['Luzon', 'Mindanao', 'Visayas']
DataFrame ['region', 'pop']
```


In [ ]:
# cities: the table from the start of Part A
# default: region becomes the INDEX
s = cities.groupby("region")["pop"].sum()
print(type(s).__name__, s.index.tolist())

# as_index=False: region stays a COLUMN
t = cities.groupby("region", as_index=False)["pop"].sum()
print(type(t).__name__, t.columns.tolist())

### Slide 21 — The same calculation with a different output shape


Expected output:

```
region
Luzon       1847000.0
Mindanao    1252500.0
Visayas      457000.0
Name: pop, dtype: float64
0    1847000.0
1    1847000.0
2    1847000.0
3     457000.0
4     457000.0
5     457000.0
6    1252500.0
7    1252500.0
Name: pop, dtype: float64
```


In [ ]:
# cities: the table from the start of Part A
print(cities.groupby("region")["pop"].median())

print(cities.groupby("region")["pop"].transform("median"))

### Slide 22 — Each project compared with its own region


Expected output:

```
budget     279,839,557
reg_median 46,771,198
vs_region  5.98
```


In [ ]:
df["reg_median"] = (df.groupby("region")
                      .budget.transform("median"))
df["vs_region"] = df.budget / df.reg_median

first = df.iloc[0]
print(f"budget     {first.budget:,.0f}")
print(f"reg_median {first.reg_median:,.0f}")
print(f"vs_region  {first.vs_region:.2f}")

### Slide 24 — Keep whole groups, not rows


Expected output:

```
4 16291
['National Capital Region', 'Region I', 'Region III', 'Region IV-A']
```


In [ ]:
big = df.groupby("region").filter(
    lambda g: len(g) > 3000)

print(big.region.nunique(), len(big))
print(sorted(big.region.unique()))

### Slide 25 — Position within a group, not overall


Expected output:

```
18
region
Central Office    1447.5
Region XI          989.1
Name: budget, dtype: float64
```


In [ ]:
df["rank_in_region"] = (df.groupby("region")
    .budget.rank(ascending=False, method="min"))

# the biggest project in each region
top = df[df.rank_in_region == 1]
print(len(top))
biggest = top.nlargest(2, "budget").set_index("region").budget
print((biggest / 1e6).round(1))     # millions of pesos

### Slide 26 — Concentration of spending within one region


Expected output:

```
1357 of 5412 projects
0.251
```


In [ ]:
r3 = (df[df.region == "Region III"]
        .sort_values("budget", ascending=False))
r3["cum_share"] = r3.budget.cumsum() / r3.budget.sum()

half = (r3.cum_share <= .5).sum() + 1
print(half, "of", len(r3), "projects")
print(round(half / len(r3), 3))

### Slide 33 — merge on the shared column


**Exercise.**

1. Predict the most densely populated city.
2. Run the cell.
3. Change ascending=False to ascending=True. Which cities are least dense, and why?


In [ ]:
# cities: the table from the start of Part A
areas = pd.DataFrame({
    "city": ["Quezon City", "Manila", "Baguio", "Cebu",
             "Iloilo", "Tacloban", "Davao", "Cagayan de Oro"],
    "area_km2": [172, 42, 58, 315, 78, 202, 2444, 413]})

both = pd.merge(cities, areas, on="city")
both["density"] = both["pop"] / both["area_km2"]
print(both.sort_values("density", ascending=False).head(3).round())

### Slide 36 — validate= converts a silent error into an exception


Expected output:

```
8 rows -> 8 rows
MergeError: Merge keys are not unique in right dataset; not a many-to-one merge
```


In [ ]:
# cities, areas: the tables from the merge exercise
ok = cities.merge(areas, on="city", validate="many_to_one")
print(len(cities), "rows ->", len(ok), "rows")

dup = pd.concat([areas, areas.head(1)])     # Quezon City twice
try:
    cities.merge(dup, on="city", validate="many_to_one")
except pd.errors.MergeError as e:
    print("MergeError:", str(e).splitlines()[0])

### Slide 37 — indicator=True labels every row


**Exercise.**

1. Predict how many cities end up left_only.
2. Run the cell.
3. Change != "Davao" to == "Davao", keeping only Davao’s area. How many are left_only now?


In [ ]:
# cities, areas: the tables from the merge exercise
some = areas[areas.city != "Davao"]     # Davao's area missing
m = cities.merge(some, on="city", how="left", indicator=True)

print(m["_merge"].value_counts())
print(m[m["_merge"] == "left_only"]["city"].tolist())

### Slide 38 — Stacking rows or columns without a warning


Expected output:

```
city    region      pop
0     Quezon City     Luzon  2960000
1          Manila     Luzon  1847000
7  Cagayan de Oro  Mindanao   728000
city  area_km2
0  Quezon City       NaN
1       Manila       NaN
2       Baguio       NaN
5          NaN     202.0
6          NaN    2444.0
7          NaN     413.0
```


In [ ]:
# cities, areas: the tables from the merge exercise
# axis=0 (default): rows stacked
print(pd.concat([cities.head(2), cities.tail(1)]))

# axis=1: columns side by side, matched on the INDEX
left = cities[["city"]].head(3)       # labels 0, 1, 2
right = areas[["area_km2"]].tail(3)   # labels 5, 6, 7
print(pd.concat([left, right], axis=1))

### Slide 44 — pivot_table spreads a category out


**Exercise.**

1. Predict the shape of the result.
2. Run the cell.
3. Add a fifth row, Luzon in 2025 with sales 5, to the three lists. What does the Luzon 2025 cell show now?


In [ ]:
sales = pd.DataFrame({"region": ["Visayas", "Visayas", "Luzon", "Luzon"],
                      "year": [2024, 2025, 2024, 2025],
                      "sales": [10, 14, 22, 25]})

wide = sales.pivot_table(
    index="region", columns="year",
    values="sales", aggfunc="sum")
print(wide)

### Slide 45 — pivot_table aggregates where pivot cannot


Expected output:

```
(18, 5)
status          Completed  On-Going
region
Central Office        121        50
Region III           4424       822
```


In [ ]:
pt = df.pivot_table(index="region", columns="status",
                    values="budget", aggfunc="count",
                    fill_value=0)
print(pt.shape)
print(pt.loc[["Central Office", "Region III"],
             ["Completed", "On-Going"]])

### Slide 46 — melt folds columns into rows


Expected output:

```
region  year  sales
0    Luzon  2024     22
1  Visayas  2024     10
2    Luzon  2025     25
3  Visayas  2025     14
```


In [ ]:
# sales, wide: the tables from the pivot_table exercise
long = wide.reset_index().melt(
    id_vars="region",
    var_name="year",
    value_name="sales")
print(long)

### Slide 47 — Wide to long for plotting


Expected output:

```
(18, 11) -> (198, 3)
18 empty region-years
```


In [ ]:
wide = df.pivot_table(index="region", columns="year",
                      values="budget", aggfunc="sum")
long = wide.reset_index().melt(
    id_vars="region", var_name="year", value_name="total")
print(wide.shape, "->", long.shape)
print(long.total.isna().sum(), "empty region-years")

### Slide 48 — A frequency table of two categories


Expected output:

```
(18, 5)
status      Completed  On-Going
region
Region III      0.817     0.152
```


In [ ]:
print(pd.crosstab(df.region, df.status).shape)

# as shares within each region
shares = pd.crosstab(df.region, df.status, normalize="index")
print(shares.loc[["Region III"], ["Completed", "On-Going"]].round(3))

### Slide 50 — Most densely populated city per region


**Exercise.**

1. Predict which region has the most densely populated city.
2. Run the cell.
3. Change .max() to .mean(). Does the order of the regions change?


In [ ]:
# cities, areas: the tables from the merge exercise
both = cities.merge(areas, on="city")
both["density"] = both["pop"] / both["area_km2"]

print(both.groupby("region")["density"].max().round())

---

*Data: DPWH flood control projects 2016–2026, mirrored by BetterGov.ph from the DPWH
transparency portal, CC0-1.0 (public domain). Served by the course API at* `/api/flood`.
